Complete regularization setup in PyTorch

Combining dropout, weight decay (AdamW), and early stopping in a training loop




In [9]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# model with dropout
class RegularizedNet(nn.Module):
  def __init__(self, dropout=0.3):
    super().__init__()
    self.layers = nn.Sequential(
        nn.Linear(784,256),
        nn.ReLU(),
        nn.Dropout(p=dropout), # dropout after activation
        nn.Linear(256,128),
        nn.ReLU(),
        nn.Dropout(p=dropout),
        nn.Linear(128,10)
    )
  def forward(self, x):
    return self.layers(x)

model = RegularizedNet(dropout=0.3)

# ============================================================
# L2 WEIGHT DECAY (use AdamW, not Adam)
# AdamW decouples weight decay from gradient updates
# for more uniform regularization
# ============================================================

optimizer = torch.optim.AdamW(model.parameters(),
                              lr = 1e-3,
                              weight_decay = 0.01) #l2 penalty strength
# EARLY STOPPING

class EarlyStopping:
  def __init__(self, patience = 5, min_delta = 0.001):
    self.patience = patience
    self.min_delta = min_delta
    self.counter = 0
    self.best_loss = float('inf')
    self.best_model = None

  def __call__(self, val_loss, model):
    if val_loss < self.best_loss - self.min_delta:
      self.best_loss = val_loss
      self.best_model = {
          k: v.clone() for k,v in model.state_dict().items()
      }
      self.counter = 0
    else:
      self.counter += 1
    return self.counter >= self.patience

  def restore_best(self, model):
    model.load_state_dict(self.best_model)

# training loop with all three techniques
early_stop = EarlyStopping(patience=7, min_delta = 0.001)
criterion = nn.CrossEntropyLoss()

# synthetic data for demonstration
train_data = TensorDataset(torch.randn(320,784), torch.randint(0, 10, (320,)))
val_data = TensorDataset(torch.randn(64,784), torch.randint(0, 10, (64,)))

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32)

for epoch in range(100):
  model.train()
  train_loss = 0
  for x, y in train_loader:
    optimizer.zero_grad()
    loss = criterion(model(x), y)
    loss.backward()
    optimizer.step()
    train_loss += loss.item()
  train_loss /= len(train_loader)

 # validation dropout disabled
  model.eval()
  val_loss = 0
  with torch.no_grad():
    for x,y in val_loader:
      val_loss += criterion(model(x), y).item()

  val_loss /= len(val_loader) #average over batches
  print(f"Epoch {epoch}: train={train_loss:.3f}, val={val_loss:.3f}")

  if(early_stop(val_loss, model)):
    print(f"Early Stopping at epoch {epoch}")
    early_stop.restore_best(model)
    break


Epoch 0: train=2.324, val=2.269
Epoch 1: train=2.100, val=2.261
Epoch 2: train=1.866, val=2.257
Epoch 3: train=1.481, val=2.274
Epoch 4: train=0.975, val=2.301
Epoch 5: train=0.474, val=2.409
Epoch 6: train=0.233, val=2.555
Epoch 7: train=0.079, val=2.639
Epoch 8: train=0.046, val=2.708
Epoch 9: train=0.033, val=2.774
Early Stopping at epoch 9
